# CodaMOSA vs CoverUp — Experiment Control

This notebook is the single observable control panel. It calls versioned scripts; it does not duplicate the experiment logic.

**Design interpretation:** File B wins. This repository runs CodaMOSA, CoverUp, and the Pynguin baseline on MT and CM. B1 = gpt-5.6-sol, B2 = gpt-5.6-luna (separate models on stream.srbyte.id). MuTAP rows come from the collaborator.

**Safety:** edit `config/local.env`, which is gitignored. Never paste keys into notebook cells or `experiment.toml`.

**Kernel:** use `.venv (Python 3.14.4)`. This notebook must be opened from inside the repository directory.


## 0. Kernel and local credentials

Use `.venv (Python 3.14.4)`. Fill `config/local.env` outside this notebook. This cell prints presence only, never values.


In [ ]:
from pathlib import Path
import sys

# Find the repo root by walking up from the current directory, and fail loudly
# if it cannot be found instead of silently falling back to "/".
_root = Path.cwd().resolve()
while not (_root / "config" / "experiment.toml").exists():
    if _root == _root.parent:
        raise RuntimeError(
            "config/experiment.toml not found above the current directory "
            f"({_root}). Open this notebook from inside the repository "
            "directory and restart the kernel before running this cell."
        )
    _root = _root.parent
ROOT = _root

_scripts = str(ROOT / "scripts")
if _scripts in sys.path:
    sys.path.remove(_scripts)
sys.path.insert(0, _scripts)

try:
    from scripts.notebook_control import load_local_env, presence, run, print_status
except ImportError:
    from notebook_control import load_local_env, presence, run, print_status

loaded = load_local_env()
print("Loaded variable names:", loaded)
print("Presence:", presence())
print("ROOT:", ROOT)
assert all(presence().values()), "Fill config/local.env before API probes."


## 1. No-cost setup and validation

Creates/updates the environment, inventories 588 tasks, builds the 80-run pilot matrix (incl. Pynguin baseline rows), and runs tests. Safe to rerun.


In [ ]:
run(["bash", "scripts/setup.sh"], timeout=1200)

## 2. Inspect dataset inventory and pilot plan

Expected totals: 163 MT functions + 425 CM modules = 588. Pilot: 5 MT + 3 disjoint CM × 2 tools × 2 backbones × R=2 = 64 runs.


In [ ]:
run([sys.executable, "scripts/inventory.py"])
run([sys.executable, "scripts/build_matrix.py", "--pilot"])
run([sys.executable, "scripts/run_experiment.py", "--pilot", "--dry-run"])
print_status()

In [ ]:
run([sys.executable, "scripts/report.py"])
from IPython.display import Markdown
Markdown((ROOT / "experiment_report.md").read_text(encoding="utf-8"))

## 3. Endpoint/tool probes — required before pilot

This validates B1/B2 `model_returned`, CoverUp function calling, and the per-call rate gate. The probe sends `tool_choice: required` (string form) because this endpoint rejects the object form. B1 = `gpt-5.6-sol`, B2 = `gpt-5.6-luna` — two distinct models on stream.srbyte.id (File B two-backbone design).


In [ ]:
probe_run = run([sys.executable, "scripts/probe_endpoints.py"], timeout=300, check=False)
probe = ROOT / "results" / "endpoint_probe.json"
probe_results = __import__("json").loads(probe.read_text(encoding="utf-8"))
for item in probe_results:
    print(item)
failed = [item for item in probe_results if not item.get("passed")]
PROBES_PASSED = not failed
if failed:
    print("\nBLOCKED: a backbone probe did not pass. Fix the failing backbone, rerun this cell, then continue to Section 4.")
    print("Probe report saved to results/endpoint_probe.json")
else:
    print("Both endpoint function-calling probes passed. Continue to pilot smoke test.")
PROBES_PASSED

## 4. Pilot — do not run until Section 3 passes

Runs 80 isolated jobs with checkpoints, 600 s budgets, configured workers, raw logs, and the USD 5,000 cost gates. This cell auto-skips when the pilot is already complete.

**Pause / resume:** The runner checkpoints every completed run to `results/pilot_results.csv`. To pause: `touch results/pause.requested` (or run `python scripts/run_experiment.py --pause`). The current in-flight run will finish, then no new runs launch. To resume: `python scripts/run_experiment.py --resume --retry-failed --pilot`. Re-running without `--retry-failed` skips already-completed runs automatically.


In [ ]:
PROBES_PASSED = globals().get("PROBES_PASSED", False)
CONFIRM_PAID_PILOT = True

import csv as _csv
_pilot = ROOT / "results" / "pilot_results.csv"
_pilot_rows = list(_csv.DictReader(_pilot.open(newline="", encoding="utf-8"))) if _pilot.exists() else []
_pilot_ok = sum(1 for r in _pilot_rows if r.get("status") == "ok")

if _pilot_rows and _pilot_ok >= 0.8 * len(_pilot_rows):
    print(f"SKIPPED: pilot already complete ({_pilot_ok}/{len(_pilot_rows)} OK). "
          "Use the dual-runner cell below for the main queues.")
elif not PROBES_PASSED:
    print("SKIPPED: Section 3 probe gate has not passed. No pilot task was started.")
elif not CONFIRM_PAID_PILOT:
    print("SKIPPED: set CONFIRM_PAID_PILOT=True after reviewing probe results.")
else:
    try:
        run([sys.executable, "scripts/run_experiment.py", "--pilot", "--retry-failed", "--max-runs", "1", "--workers", "1"], timeout=2400)
        print_status()
    except RuntimeError as exc:
        print(f"\nSmoke run failed: {exc}")
        print("Aborting remaining pilot runs. Inspect results/generated/ and results/raw/ for the failure.")
        raise
    run([sys.executable, "scripts/run_experiment.py", "--pilot", "--retry-failed"], timeout=172800)


## 5. Pilot monitoring and analysis

Rerun while the pilot is active or after interruption. The runner is resumable — every completed run is checkpointed, so re-running skips what is already done.

**Resume commands:**
- `python scripts/run_experiment.py --resume --pilot` (skips completed runs)
- `python scripts/run_experiment.py --resume --retry-failed --pilot` (also re-runs failed runs)
- `python scripts/run_experiment.py --pause` (pause after current run finishes)


In [ ]:
print_status()
path = ROOT / "results" / "pilot_results.csv"
if path.exists():
    run([sys.executable, "scripts/analyze.py", str(path), "--output", "results/pilot_analysis"])
    import csv
    from collections import Counter
    rows = list(csv.DictReader(path.open(newline="", encoding="utf-8")))
    total = len(rows)
    ok = sum(1 for r in rows if r.get("status") == "ok")
    failed = sum(1 for r in rows if r.get("status") == "failed")
    cost = sum(float(r.get("cost_usd") or 0) for r in rows)
    by_cfg = Counter((r.get("suite"), r.get("tool"), r.get("backbone")) for r in rows)
    print(f"\n=== PILOT SUMMARY: {total} runs ===")
    print(f"OK: {ok} | Failed: {failed} | Total cost: ${cost:.6f}")
    print("\nConfiguration breakdown:")
    for (suite, tool, backbone), count in sorted(by_cfg.items()):
        cfg_ok = sum(1 for r in rows if r.get("suite")==suite and r.get("tool")==tool and r.get("backbone")==backbone and r.get("status")=="ok")
        print(f"  {suite}/{tool}/{backbone}: {count} runs, {cfg_ok} OK")
else:
    print("SKIPPED: no pilot_results.csv yet.")

## 6. Full run status

Report-only: the main execution (10,150 runs incl. the Pynguin baseline) is complete and checkpointed. Launching of any remaining queues is handled by the runner cells in Section 6b.


In [ ]:
import csv
import tomllib

pilot_path = ROOT / "results" / "pilot_results.csv"
pilot_rows = list(csv.DictReader(pilot_path.open(newline="", encoding="utf-8"))) if pilot_path.exists() else []
pilot_ok = sum(1 for r in pilot_rows if r.get("status") == "ok")

results_path = ROOT / "results" / "results.csv"
result_rows = list(csv.DictReader(results_path.open(newline="", encoding="utf-8"))) if results_path.exists() else []
spent = sum(float(r.get("cost_usd") or 0) for r in result_rows)
config = tomllib.loads((ROOT / "config" / "experiment.toml").read_text(encoding="utf-8"))["experiment"]
cost_cap = float(config["cost_cap_usd"])
launch_cap = cost_cap * float(config["stop_launch_fraction"])
print(f"Pilot: {pilot_ok}/{len(pilot_rows)} OK | LLM spent: USD {spent:.2f} / launch cap {launch_cap:.2f}")
print("Launching is handled by the dual-runner cell below (Pynguin + LLM in parallel, separate checkpoint files).")


## 6b. Runner terpasang (attached) — pantau langsung di cell

Satu cell di bawah menjalankan **B2 (glm-5.3-flash, RQ2) + Pynguin secara paralel** dengan output live:
prefiks `[B2]` = glm-5.3-flash (router.octalabs.id), prefiks `[PYG]` = baseline Pynguin (gratis).

- **Berhenti:** tombol Stop/Interrupt (■) VS Code — checkpoint aman, jalankan ulang cell untuk lanjut
- **Jangan tutup VS Code** selama berjalan — proses ikut mati (checkpoint tetap aman, ulangi cell)
- Runner retry-LLM-lama sengaja tidak disertakan (run_id bentrok dengan B2; datanya deterministik)
- **Prasyarat:** jalankan dulu cell "0. Kernel and local credentials"


In [ ]:
# Runner TERPASANG (attached): B2-glm53 (RQ2) + Pynguin jalan PARALEL,
# output langsung mengalir ke cell ini dengan prefiks [B2] / [PYG].
# Cell tetap sibuk sampai selesai — PANTAU langsung di sini.
# Berhenti: tekan tombol Stop/Interrupt (■) di VS Code — checkpoint aman, ulangi cell untuk lanjut.
# JANGAN tutup VS Code/notebook selama berjalan: proses ikut mati (checkpoint tetap aman).
import subprocess
import threading

RUNNERS = {
    "B2": [sys.executable, "-u", "scripts/run_experiment.py",
           "--matrix", "results/run_matrix_b2luna.csv",
           "--results", "results/b2_luna_results.csv",
           "--backbone", "B2", "--retry-failed", "--workers", "3"],
    "PYG": [sys.executable, "-u", "scripts/run_experiment.py",
            "--tool", "Pynguin", "--results", "results/pynguin_results.csv",
            "--retry-failed", "--workers", "3"],
}

procs = {tag: subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
         for tag, cmd in RUNNERS.items()}
print(f"Berjalan: B2-glm53 PID {procs['B2'].pid} (-> results/b2_luna_results.csv) | "
      f"Pynguin PID {procs['PYG'].pid} (-> results/pynguin_results.csv)")
print("=" * 70)

def pump(tag, proc):
    for line in proc.stdout:
        print(f"[{tag}] {line}", end="", flush=True)

threads = [threading.Thread(target=pump, args=(t, p), daemon=True) for t, p in procs.items()]
for t in threads:
    t.start()
try:
    for t in threads:
        t.join()
    print("\nSemua runner selesai. Ulangi cell ini kapan pun untuk melanjutkan dari checkpoint.")
except KeyboardInterrupt:
    print("\nDIHENTIKAN — membunuh kedua runner (checkpoint tetap aman)...")
    for p in procs.values():
        p.kill()

## 7. Final analysis and figures

Generates Tables IV–IX, paired tests, cost-effectiveness, failure rate, exclusion audit, and Figures 3–5 from real `results.csv`. Runs after the full run completes.


In [ ]:
import csv

results = ROOT / "results" / "results.csv"
if not results.exists():
    print("SKIPPED: no final results.csv yet.")
else:
    with results.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    n_ok = sum(1 for r in rows if r.get("status") == "ok")
    if n_ok == 0:
        print(f"SKIPPED: results.csv has {len(rows)} rows but 0 with status=ok.")
    else:
        analysis = run([sys.executable, "scripts/analyze.py", str(results)], check=False)
        figures = run([sys.executable, "scripts/figures.py", str(results)], check=False)
        if analysis.returncode or figures.returncode:
            print("Analysis is incomplete; inspect the command output above. Existing artifacts were preserved.")
        else:
            print(f"Analysis ready: results/analysis and results/figures ({n_ok}/{len(rows)} OK runs).")
run([sys.executable, "scripts/report.py"])
report = ROOT / "experiment_report.md"
print(f"Report ready: {report.relative_to(ROOT)} ({len(report.read_text(encoding='utf-8').splitlines())} lines).")
